# First steps with the new code
Lets try the new Trefftzmesh

In [1]:
from trefftz.mesh import TrefftzMesh

In [2]:
from enum import IntEnum, StrEnum



class WaveguideRegionsGMSH(IntEnum):
    OMEGA = 0
    GAMMA = 1
    SIGMA_L = 2
    SIGMA_R = 3


def CleanWaveguide_GMSH(H: float = 1., R: float = 5., lc: float = 0.3, verbose: bool = False) -> TrefftzMesh[WaveguideRegionsGMSH]:
    '''Creates a domain corresponging to a waveguide without scatterers.
    It assumes the default tags for the subregions, i.e.:
    - Omega = 0
    - Gamma = 1
    - Sigma = 2
    '''
    import gmsh


    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", int(verbose))
    gmsh.model.add("Waveguide")
    p0 = gmsh.model.geo.addPoint(-R, 0., 0., lc)
    p1 = gmsh.model.geo.addPoint( R, 0., 0., lc)
    p2 = gmsh.model.geo.addPoint( R,  H, 0., lc)
    p3 = gmsh.model.geo.addPoint(-R,  H, 0., lc)

    bottom = gmsh.model.geo.addLine(p0, p1)
    right  = gmsh.model.geo.addLine(p1, p2)
    top    = gmsh.model.geo.addLine(p2, p3)
    left   = gmsh.model.geo.addLine(p3, p0)

    boundary = gmsh.model.geo.addCurveLoop([bottom, right, top, left])
    domain = gmsh.model.geo.addPlaneSurface([boundary])
    gmsh.model.geo.synchronize()

    gmsh.model.addPhysicalGroup(2, [domain], WaveguideRegionsGMSH.OMEGA, "Omega")
    gmsh.model.addPhysicalGroup(1, [bottom, top], WaveguideRegionsGMSH.GAMMA, "Gamma")
    gmsh.model.addPhysicalGroup(1, [left], WaveguideRegionsGMSH.SIGMA_L, "Sigma_L")
    gmsh.model.addPhysicalGroup(1, [right], WaveguideRegionsGMSH.SIGMA_R, "Sigma_R")
    
    gmsh.model.geo.synchronize()
    gmsh.model.mesh.generate(2)
 
    mesh = TrefftzMesh.from_gmsh(gmsh.model, boundary_regions = WaveguideRegionsGMSH) #gmsh needs to be initialized for this to work
    
    gmsh.finalize()

    return mesh


class WaveguideRegionsNGSOLVE(StrEnum):
    OMEGA = "Omega"
    GAMMA = "Gamma"
    SIGMA_L = "Sigma_L"
    SIGMA_R = "Sigma_R"

def CleanWaveguide_NGSOLVE(H: float = 1., R: float = 5., lc: float = 0.3, verbosity: int = 0) -> TrefftzMesh[WaveguideRegionsNGSOLVE]:

    '''Creates a domain corresponging to a waveguide without scatterers.
    It assumes the default tags for the subregions, i.e.:
    - Omega = 0
    - Gamma = 1
    - Sigma = 2
    '''
    from netgen.geom2d import SplineGeometry
    from ngsolve import Mesh
    geo = SplineGeometry()
    p0 = geo.AddPoint(-R, 0.)
    p1 = geo.AddPoint( R, 0.)
    p2 = geo.AddPoint( R, H)
    p3 = geo.AddPoint(-R, H)
 
    bottom = geo.Append(["line", p0, p1], bc=WaveguideRegionsNGSOLVE.GAMMA)
    right  = geo.Append(["line", p1, p2], bc=WaveguideRegionsNGSOLVE.SIGMA_R)
    top    = geo.Append(["line", p2, p3], bc=WaveguideRegionsNGSOLVE.GAMMA)
    left   = geo.Append(["line", p3, p0], bc=WaveguideRegionsNGSOLVE.SIGMA_L)


    ngmesh = Mesh(geo.GenerateMesh(maxh=lc, perfstepsend=verbosity))

    
    mesh = TrefftzMesh.from_ngsolve(ngmesh, boundary_regions=WaveguideRegionsNGSOLVE)

    return mesh



In [3]:
R = 0.5
H = 1. 

NGSOLVE = True

if NGSOLVE:
    mesh = CleanWaveguide_NGSOLVE(R=R, H=H, verbosity=0, lc=0.5)
    WaveguideRegions = WaveguideRegionsNGSOLVE
else:
    mesh = CleanWaveguide_GMSH(R=R, H=H, verbose=True, lc=0.5)
    WaveguideRegions = WaveguideRegionsGMSH


In [4]:
from trefftz.mesh.checking_utilities import explore_edges

In [ ]:
import matplotlib
# matplotlib.use("TkAgg")
# print(matplotlib.get_backend())

TkAgg


In [6]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()

In [7]:
explore_edges(mesh,figax=(fig, ax))

In [8]:
from problems.base2 import Problem

SyntaxError: expected ':' (base2.py, line 61)

In [ ]:
from trefftz.dg.basis import LinearlySpacedBasis

k = 8.0

basis = LinearlySpacedBasis(N_elements=mesh.n_triangles, k=k, N_theta=10)

In [ ]:
from problems.base2 import SoundHardBC, NtDBC

In [ ]:
boundary_conditions = {WaveguideRegions.GAMMA: SoundHardBC(),
                       WaveguideRegions.SIGMA_L: NtDBC(truncating_radius=R, data=1.),
                       WaveguideRegions.SIGMA_R: NtDBC(truncating_radius=R, data=1.)}

In [41]:
from problems.base2 import SerialNumerics

In [42]:
from trefftz.dg.serial_kernels import SoundHardKernel, UltraWeakKernel, NtDLocal, WaveguideNtD_nonlocal

In [43]:
serial_numerics = SerialNumerics(interior_kernel=UltraWeakKernel(a=0.5, b=0.5),
                                 local_boundary_kernels={SoundHardBC: SoundHardKernel(d_1 = 0.5),
                                                         NtDBC: NtDLocal(R=R, d_2=0.5, H=H, n=1)},
                                 nonlocal_boundary_kernels={NtDBC: WaveguideNtD_nonlocal(R=R, d_2=0.5, M = 20, H = 1.)})

In [44]:
P = Problem(mesh=mesh, wavenumber=k, basis=basis, boundary_conditions=boundary_conditions, numerics=serial_numerics)

In [45]:
A = P.assemble_LHS()

In [46]:
import matplotlib.pyplot as plt 
plt.spy(A)

In [47]:
b = P.assemble_RHS()

In [48]:
plt.plot(P.b.real)

AttributeError: 'NoneType' object has no attribute 'real'

In [39]:
SoundHardBC().data is None

True

In [50]:
P.regions_local_kernel

[<WaveguideRegionsNGSOLVE.GAMMA: 'Gamma'>,
 <WaveguideRegionsNGSOLVE.SIGMA_L: 'Sigma_L'>,
 <WaveguideRegionsNGSOLVE.SIGMA_R: 'Sigma_R'>]